# Exploración Inicial — Powerlifting Analytics
### Fuentes: SQL (Gimnasio) + CSV (Kaggle) + API (wger)

## 1. Cargar las 3 fuentes de datos

In [ ]:
import pandas as pd
import sqlite3
import requests
import matplotlib.pyplot as plt

# ── Fuente 1: SQL (datos propios del gimnasio) ──
conn_gym = sqlite3.connect('data/gimnasio.db')
df_gimnasio = pd.read_sql_query('SELECT * FROM atletas_gimnasio', conn_gym)
conn_gym.close()
print(f'Gimnasio (SQL): {df_gimnasio.shape[0]} atletas, {df_gimnasio.shape[1]} columnas')
display(df_gimnasio.head())

In [ ]:
# ── Fuente 2: CSV (historial mundial Kaggle) ──
df_atletas = pd.read_csv('data/openpowerlifting.csv', low_memory=False)
df_meets   = pd.read_csv('data/meets.csv')
df_csv = pd.merge(df_atletas, df_meets, on='MeetID', how='left')
print(f'Dataset mundial (CSV): {df_csv.shape[0]} filas, {df_csv.shape[1]} columnas')
display(df_csv.head())

In [ ]:
# ── Fuente 3: API wger ──
BASE = 'https://wger.de/api/v2'
musculos   = requests.get(f'{BASE}/muscle/?format=json').json()['results']
equipment  = requests.get(f'{BASE}/equipment/?format=json').json()['results']
categorias = requests.get(f'{BASE}/exercisecategory/?format=json').json()['results']

df_musculos   = pd.DataFrame(musculos)[['id','name_en']].rename(columns={'name_en':'musculo'})
df_equipment  = pd.DataFrame(equipment)[['id','name']].rename(columns={'name':'equipamiento'})
df_categorias = pd.DataFrame(categorias)[['id','name']].rename(columns={'name':'categoria'})

# Mapeo manual de los 3 ejercicios del powerlifting
df_ejercicios = pd.DataFrame({
    'ejercicio':   ['Squat',        'Bench',      'Deadlift'],
    'musculo':     ['Quadriceps',   'Pectorals',  'Hamstrings'],
    'equipamiento':['Barbell',      'Barbell',    'Barbell'],
    'categoria':   ['Strength',     'Strength',   'Strength'],
    'dificultad':  ['Expert',       'Expert',     'Expert']
})
print('Datos API wger:')
display(df_ejercicios)

## 2. EDA — Fuente SQL (Gimnasio propio)

In [ ]:
print('Columnas del gimnasio:')
print(df_gimnasio.columns.tolist())
print(f'\nNaN por columna:')
print(df_gimnasio.isnull().sum())

In [ ]:
# Distribución de niveles en el gimnasio
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_gimnasio['nivel_atleta'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Niveles de atletas en el gimnasio')
axes[0].set_xlabel('Nivel')

df_gimnasio['ejercicio_principal'].value_counts().plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Ejercicio principal')
axes[1].set_xlabel('Ejercicio')

df_gimnasio['plan_membresia'].value_counts().plot(kind='bar', ax=axes[2], color='green')
axes[2].set_title('Planes de membresía')
axes[2].set_xlabel('Plan')

plt.tight_layout()
plt.show()

## 3. EDA — Fuente CSV (Historial mundial Kaggle)

In [ ]:
print('Columnas del dataset mundial:')
print(df_csv.columns.tolist())
print(f'\nNaN por columna (top 10):')
print(df_csv.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Distribución de los mejores levantamientos (sin negativos)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df_csv[df_csv['BestSquatKg'] > 0]['BestSquatKg'].dropna().plot(
    kind='hist', bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Distribución BestSquatKg')

df_csv[df_csv['BestBenchKg'] > 0]['BestBenchKg'].dropna().plot(
    kind='hist', bins=50, ax=axes[1], color='darkorange')
axes[1].set_title('Distribución BestBenchKg')

df_csv[df_csv['BestDeadliftKg'] > 0]['BestDeadliftKg'].dropna().plot(
    kind='hist', bins=50, ax=axes[2], color='green')
axes[2].set_title('Distribución BestDeadliftKg')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 países con más atletas
top_paises = df_csv['MeetCountry'].value_counts().head(10)
top_paises.plot(kind='bar', figsize=(10, 4), color='steelblue')
plt.title('Top 10 países con más competidores')
plt.xlabel('País')
plt.ylabel('Cantidad de atletas')
plt.tight_layout()
plt.show()

In [ ]:
# Promedio por sexo
promedios = df_csv.groupby('Sex')[['BestSquatKg','BestBenchKg','BestDeadliftKg']].mean()
print('Promedio por sexo:')
display(promedios)

## 4. EDA — Fuente API (wger)

In [ ]:
print('Músculos disponibles en la API:')
display(df_musculos)
print('\nEquipamiento disponible:')
display(df_equipment)
print('\nCategorías disponibles:')
display(df_categorias)

## 5. EDA Final — Las 3 fuentes combinadas

In [ ]:
# Cargar el dataset final del ETL
conn_final = sqlite3.connect('data/dataset_final.db')
df_final = pd.read_sql_query('SELECT * FROM vista_dashboard', conn_final)
conn_final.close()
print(f'Dataset final combinado: {df_final.shape[0]} filas, {df_final.shape[1]} columnas')
display(df_final.head())

In [ ]:
# Promedio de BestSquatKg por músculo (dato que solo existe al combinar las 3 fuentes)
df_validos = df_final[df_final['BestSquatKg'] > 0]
promedio_musculo = df_validos.groupby('muscle')['BestSquatKg'].mean().sort_values(ascending=False)

promedio_musculo.plot(kind='bar', figsize=(8, 4), color='steelblue')
plt.title('Promedio BestSquatKg por músculo trabajado')
plt.xlabel('Músculo')
plt.ylabel('Kg promedio')
plt.tight_layout()
plt.show()

In [ ]:
# Nivel del atleta en el gimnasio vs su rendimiento mundial
df_validos2 = df_final[df_final['TotalKg'] > 0]
nivel_rendimiento = df_validos2.groupby('nivel_atleta')['TotalKg'].mean().sort_values(ascending=False)

nivel_rendimiento.plot(kind='bar', figsize=(8, 4), color='darkorange')
plt.title('TotalKg promedio por nivel del gimnasio')
plt.xlabel('Nivel')
plt.ylabel('TotalKg promedio')
plt.tight_layout()
plt.show()